# 01 · Alertas
Lê a aba CONTROLE SCCON (camada bronze), padroniza e valida os alertas
e grava `alertas.parquet` na camada prata. Um alerta por linha.

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# Caminhos do projeto (relativos à pasta notebooks/)
bronze_dir = Path("../dados/bronze")
silver_dir = Path("../dados/prata")
source_file = bronze_dir / "Controle de alertas - CMF-SEAMA.xlsx"   # troque pelo nome real

In [ ]:
# Lê a aba de alertas da SCCON
df_control = pd.read_excel(source_file, sheet_name="CONTROLE SCCON")

# Colunas que seguem para a camada prata, com nomes padronizados
control_columns = {
    "idt_alerta": "id_alerta",
    "tipo": "tipo_alerta",
    "cidade do alerta": "municipio",
    "area_ha": "area_ha",
    "dat_antes": "data_imagem_antes",
    "dat_depois": "data_imagem_depois",
    "publicação": "data_publicacao",
    "lat_centro": "latitude",
    "long_centr": "longitude",
    "Responsável": "unidade_responsavel",
    "resposta": "tem_resposta",
}
df_alerts = df_control[list(control_columns)].rename(columns=control_columns)

In [ ]:
# Converte coordenadas em graus, minutos e segundos (ex.: 18°53'9.2"S) para graus decimais
def dms_to_decimal(series):
    pattern = r"(?P<deg>\d+)\D+(?P<min>\d+)\D+(?P<sec>\d+(?:\.\d+)?)\D*(?P<hem>[NSEW])"
    parts = series.astype("string").str.extract(pattern)
    decimal = (parts["deg"].astype(float)
               + parts["min"].astype(float) / 60
               + parts["sec"].astype(float) / 3600)
    # Sul e Oeste são negativos
    return decimal.where(~parts["hem"].isin(["S", "W"]), -decimal)

In [ ]:
# Limpeza
df_alerts = df_alerts.dropna(subset=["id_alerta"])  # remove a linha vazia
df_alerts["id_alerta"] = df_alerts["id_alerta"].astype("int64")
df_alerts["municipio"] = (df_alerts["municipio"]
                          .str.replace(r"\s*-\s*ES$", "", regex=True)
                          .str.strip())
df_alerts["tem_resposta"] = df_alerts["tem_resposta"].str.strip().str.lower().eq("sim")

# Coordenadas: de graus/minutos/segundos para decimal
df_alerts["latitude"] = dms_to_decimal(df_alerts["latitude"])
df_alerts["longitude"] = dms_to_decimal(df_alerts["longitude"])

# Área: vírgula decimal vira ponto antes de converter
df_alerts["area_ha"] = pd.to_numeric(
    df_alerts["area_ha"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce")

In [ ]:
# Checagens de qualidade: se alguma falhar, o notebook para aqui e diz o motivo
assert df_alerts["id_alerta"].is_unique, "Há alertas repetidos"
assert df_alerts["latitude"].dropna().between(-21.4, -17.8).all(), "Latitude fora do ES"
assert df_alerts["longitude"].dropna().between(-41.9, -39.6).all(), "Longitude fora do ES"
assert (df_alerts["area_ha"].dropna() > 0).all(), "Área zero ou negativa"

In [ ]:
# Salva a tabela na camada prata
silver_dir.mkdir(parents=True, exist_ok=True)
df_alerts.to_parquet(silver_dir / "alertas.parquet", index=False)

print(len(df_alerts), "alertas salvos")
df_alerts.info()